# Random Forest com Gradient Boosting


---
## 1. 🔄 Recapitulação: Bagging vs. Boosting

| Aspecto               | Bagging (Random Forest)              | Boosting (GBM)                         |
|-----------------------|--------------------------------------|----------------------------------------|
| Treinamento           | Paralelo (independente)              | **Sequencial** (cada modelo corrige o anterior) |
| Foco de cada modelo   | Amostra aleatória dos dados          | **Exemplos difíceis** (maior peso/resíduo) |
| Reduz principalmente  | Variância                            | **Viés** (e variância controlada)      |
| Risco de overfitting  | Baixo (naturalmente)                 | Médio-alto (requer regularização)      |
| Velocidade treino     | Rápido (paralelizável)               | Mais lento (sequencial)                |
| Performance típica    | Excelente                            | **Estado da arte** em dados tabulares  |

> **Intuição do Boosting:** Em vez de treinar todos os modelos igualmente, cada modelo foca nos **erros** que os anteriores cometeram. É como uma equipe onde cada membro resolve o problema que a equipe ainda não sabe resolver.

---
## 2. 🎯 AdaBoost — Boosting Clássico

**AdaBoost** (Freund & Schapire, 1996) foi o primeiro algoritmo de boosting prático.

### Algoritmo

1. Inicialize pesos uniformes: $w_i = \frac{1}{n}$ para todos os $i$

2. Para $b = 1, ..., B$:
   - Treine classificador $h_b$ com pesos $w_i$
   - Calcule erro ponderado: $\varepsilon_b = \sum_{i: h_b(x_i) \neq y_i} w_i$
   - Calcule contribuição: $\alpha_b = \frac{1}{2} \ln\left(\frac{1 - \varepsilon_b}{\varepsilon_b}\right)$
   - Atualize pesos: $w_i \leftarrow w_i \cdot \exp(-\alpha_b \cdot y_i \cdot h_b(x_i))$
   - Normalize: $w_i \leftarrow w_i / \sum_j w_j$

3. Predição final: $H(x) = \text{sign}\left(\sum_{b=1}^{B} \alpha_b h_b(x)\right)$

**Limitação:** Sensível a ruído e outliers (que recebem pesos muito grandes).

In [ ]:
# ─── Instalação (se necessário no Colab) ─────────────────────────────────────
# !pip install xgboost lightgbm scikit-learn matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (load_iris, load_breast_cancer, load_diabetes,
                               make_classification, make_regression)
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               AdaBoostClassifier,
                               GradientBoostingClassifier, GradientBoostingRegressor,
                               HistGradientBoostingClassifier,
                               HistGradientBoostingRegressor)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay,
                              mean_squared_error, r2_score, accuracy_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

try:
    import xgboost as xgb
    print("XGBoost disponível ✅")
except ImportError:
    print("XGBoost não instalado. Execute: !pip install xgboost -q")

try:
    import lightgbm as lgb
    print("LightGBM disponível ✅")
except ImportError:
    print("LightGBM não instalado. Execute: !pip install lightgbm -q")

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print("\n✅ Setup concluído!")

In [ ]:
# ─── Demo AdaBoost: visualizando pesos ───────────────────────────────────────
from sklearn.datasets import make_circles
from matplotlib.colors import ListedColormap

X_circ, y_circ = make_circles(n_samples=200, noise=0.15, factor=0.5, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_circ, y_circ, test_size=0.3, random_state=42)

n_stages = [1, 5, 10, 50]
cmap_bg  = ListedColormap(['#BBDEFB', '#FFCCBC'])
cmap_pt  = ListedColormap(['#1565C0', '#BF360C'])

h  = 0.04
xx, yy = np.meshgrid(np.arange(X_circ[:,0].min()-0.5, X_circ[:,0].max()+0.5, h),
                     np.arange(X_circ[:,1].min()-0.5, X_circ[:,1].max()+0.5, h))

fig, axes = plt.subplots(1, len(n_stages), figsize=(18, 4))
for ax, n in zip(axes, n_stages):
    ada = AdaBoostClassifier(n_estimators=n, algorithm='SAMME', random_state=42)
    ada.fit(X_tr, y_tr)
    Z = ada.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X_te[:,0], X_te[:,1], c=y_te, cmap=cmap_pt,
               edgecolors='white', s=40, lw=0.5)
    acc = ada.score(X_te, y_te)
    ax.set_title(f'AdaBoost n={n}\nAcc={acc:.3f}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('AdaBoost em Círculos: Evolução com Número de Estimadores', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. 📐 Gradient Boosting Machines (GBM) — Teoria Completa

**Gradient Boosting** (Friedman, 2001) é uma generalização do boosting que enquadra o problema como **otimização de uma função de perda** no espaço funcional.

### Ideia Central

Queremos encontrar $F^*(x) = \arg\min_F \mathbb{E}[L(y, F(x))]$

Como esse problema é intratável diretamente, usamos **gradiente descendente funcional**:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

onde $h_m$ é um modelo fraco treinado para aproximar o **pseudo-resíduo** (gradiente negativo da perda):

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$

### Função de Perda e Pseudo-Resíduos

| Tarefa          | Função de Perda $L$                         | Pseudo-resíduo $r_i$                     |
|-----------------|---------------------------------------------|------------------------------------------|
| Regressão MSE   | $\frac{1}{2}(y_i - F)^2$                    | $y_i - F(x_i)$ (resíduo simples!)        |
| Regressão MAE   | $|y_i - F|$                                 | $\text{sign}(y_i - F(x_i))$             |
| Classificação   | $\log(1 + e^{-2y_iF})$                      | $\frac{2y_i}{1+e^{2y_iF}}$              |

> **Insight:** Para MSE, o pseudo-resíduo é simplesmente o erro — o próximo modelo tenta corrigir o que o anterior errou!

---
## 4. 🌲 Gradient Boosting com Árvores (GBDT)

O modelo padrão usa **árvores de regressão rasas** como modelos fracos (base learners).

### Algoritmo Completo (Classificação Binária)

```
Inicializar F_0(x) = log(p / (1-p))  [log-odds da classe positiva]

Para m = 1 até M:
  1. Calcule pseudo-resíduos:
     r_im = y_i - p_i,   onde p_i = σ(F_{m-1}(x_i))

  2. Ajuste árvore h_m aos pares {(x_i, r_im)}
     (árvore com J folhas, profundidade típica 3-8)

  3. Para cada folha j, calcule o valor ótimo γ_jm:
     γ_jm = Σ r_im / Σ p_im(1 - p_im)   [Newton step]

  4. Atualize:
     F_m(x) = F_{m-1}(x) + η * h_m(x)

Retorne F_M(x);  predição: σ(F_M(x))
```

### Por que árvores rasas?
- Cada árvore captura interações de baixa ordem (geralmente 2-4 variáveis)
- Modelos fracos individuais → controle de overfitting
- Muitas árvores sequenciais compensam a limitação individual

In [ ]:
# ─── Visualização didática: Gradient Boosting passo a passo ──────────────────
# Regressão simples: f(x) = sin(x) + ruído

np.random.seed(42)
n_pts = 80
X_sin = np.sort(np.random.uniform(0, 2*np.pi, n_pts)).reshape(-1, 1)
y_sin = np.sin(X_sin.ravel()) + np.random.normal(0, 0.2, n_pts)

# Treinando iterativamente e mostrando os resíduos
from sklearn.tree import DecisionTreeRegressor

eta   = 0.5
etapas = [1, 3, 5, 20]
F     = np.full(n_pts, y_sin.mean())  # F_0
arvores = []

fig, axes = plt.subplots(2, len(etapas), figsize=(18, 9))
X_plot = np.linspace(0, 2*np.pi, 500).reshape(-1, 1)

for etapa in range(1, max(etapas) + 1):
    residuos = y_sin - F
    h = DecisionTreeRegressor(max_depth=3)
    h.fit(X_sin, residuos)
    arvores.append(h)
    F = F + eta * h.predict(X_sin)

    if etapa in etapas:
        idx = etapas.index(etapa)

        # Predição acumulada
        F_plot = np.full(500, y_sin.mean())
        for a in arvores:
            F_plot += eta * a.predict(X_plot)

        # Linha do seno real
        axes[0, idx].scatter(X_sin, y_sin, s=20, alpha=0.6, color='#607D8B', label='dados')
        axes[0, idx].plot(X_plot, np.sin(X_plot), 'k--', lw=1.5, label='f(x) real')
        axes[0, idx].plot(X_plot, F_plot, 'r-',  lw=2,   label=f'GBM (m={etapa})')
        axes[0, idx].set_title(f'Predição Acumulada (m={etapa})')
        axes[0, idx].legend(fontsize=8)

        # Resíduos
        res_atual = y_sin - F
        axes[1, idx].bar(range(len(res_atual)), res_atual,
                         color=['#F44336' if r < 0 else '#4CAF50' for r in res_atual],
                         width=0.8, alpha=0.7)
        rmse_atual = np.sqrt(np.mean(res_atual**2))
        axes[1, idx].axhline(0, color='black', lw=1)
        axes[1, idx].set_title(f'Resíduos (RMSE={rmse_atual:.3f})')
        axes[1, idx].set_xlabel('Amostra')
        if idx == 0:
            axes[1, idx].set_ylabel('Resíduo')

plt.suptitle('Gradient Boosting Passo a Passo: Predição e Resíduos', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. ⚙️ Regularização e Hiperparâmetros

### GradientBoostingClassifier / Regressor (scikit-learn)

| Parâmetro           | Papel                                           | Dica Prática                    |
|---------------------|-------------------------------------------------|---------------------------------|
| `n_estimators`      | Número de árvores (M)                           | 100–500; use early stopping     |
| `learning_rate` (η) | Encolhimento de cada contribuição               | 0.01–0.3; menor η → mais árvores|
| `max_depth`         | Profundidade de cada árvore base                | 3–6 (rasas funcionam bem)       |
| `subsample`         | Fração das amostras por árvore (Stochastic GB)  | 0.5–0.8 melhora generalização   |
| `max_features`      | Fração de features por split                    | Similar ao RF                   |
| `min_samples_leaf`  | Mínimo de amostras em folha                     | Regularização adicional         |

### Tradeoff Learning Rate × n_estimators

$$\text{Desempenho} \approx f\left(\eta \times M\right)$$

Reduzir η pela metade → dobrar M para manter desempenho (mas custo computacional dobra).

**Recomendação:** Comece com `learning_rate=0.1, n_estimators=100`; depois use early stopping para calibrar.

In [ ]:
# ─── Efeito do Learning Rate ──────────────────────────────────────────────────
iris  = load_iris()
X, y  = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                           stratify=y, random_state=42)

lrs    = [0.5, 0.1, 0.05, 0.01]
n_est  = 200
colors = ['#F44336', '#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, cor in zip(lrs, colors):
    gbm = GradientBoostingClassifier(n_estimators=n_est, learning_rate=lr,
                                      max_depth=3, random_state=42)
    gbm.fit(X_tr, y_tr)

    # Acurácia por iteração (staged_predict)
    train_acc = [accuracy_score(y_tr, pred)
                 for pred in gbm.staged_predict(X_tr)]
    test_acc  = [accuracy_score(y_te, pred)
                 for pred in gbm.staged_predict(X_te)]

    axes[0].plot(range(1, n_est+1), train_acc, '-', color=cor, alpha=0.8,
                 label=f'η={lr} (treino)')
    axes[1].plot(range(1, n_est+1), test_acc,  '-', color=cor, alpha=0.8,
                 label=f'η={lr}')

for ax, titulo in zip(axes, ['Acurácia Treino', 'Acurácia Teste']):
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('Acurácia')
    ax.set_title(titulo)
    ax.legend(fontsize=9)

plt.suptitle('Efeito do Learning Rate no GBM (Iris)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. ⚡ Implementações Modernas

### XGBoost (Chen & Guestrin, 2016)

Melhorias sobre o GBM padrão:
- **Regularização L1 + L2** nas folhas da árvore
- **Newton Boosting**: usa segunda derivada (Hessiana) para passos mais precisos
- **Sparsidade**: suporte nativo para dados esparsos e valores ausentes
- **Column Subsampling** (por árvore, por profundidade ou por nó)
- **Paralelismo** na construção de árvores (splits paralelos)

### LightGBM (Microsoft, 2017)

- **GOSS** (Gradient-based One-Side Sampling): mantém amostras com grande gradiente
- **EFB** (Exclusive Feature Bundling): comprime features mutuamente exclusivas
- **Crescimento folha-por-folha** (leaf-wise) vs. nível-por-nível (level-wise)
- Muito mais rápido para datasets grandes

### HistGradientBoosting (scikit-learn ≥ 0.21)

- Inspirado no LightGBM, nativo do scikit-learn
- Discretiza features em histogramas (256 bins)
- Suporte nativo a valores ausentes
- Excelente para datasets médio-grandes

In [ ]:
# ─── Comparação de Implementações ─────────────────────────────────────────────
import time

bc = load_breast_cancer()
Xb, yb = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25,
                                                stratify=yb, random_state=42)

modelos = {
    'RandomForest':         RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'GBM (sklearn)':        GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                        max_depth=4, random_state=42),
    'HistGBM (sklearn)':    HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                                            random_state=42),
}

# Adicionar XGBoost e LightGBM se disponíveis
try:
    modelos['XGBoost'] = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1,
                                            max_depth=4, use_label_encoder=False,
                                            eval_metric='logloss',
                                            random_state=42, n_jobs=-1)
except: pass

try:
    modelos['LightGBM'] = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1,
                                              max_depth=4, random_state=42,
                                              n_jobs=-1, verbose=-1)
except: pass

resultados = []
for nome, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(Xb_tr, yb_tr)
    tempo = time.time() - t0
    acc   = modelo.score(Xb_te, yb_te)
    cv5   = cross_val_score(modelo, Xb, yb, cv=5, n_jobs=-1).mean()
    resultados.append({'Modelo': nome, 'Acc Teste': acc,
                       'CV-5 Mean': cv5, 'Tempo (s)': round(tempo, 3)})

df_res = pd.DataFrame(resultados).set_index('Modelo')
print(df_res.sort_values('Acc Teste', ascending=False).to_string())

In [ ]:
# ─── Gráfico comparativo ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_sorted = df_res.sort_values('Acc Teste', ascending=True)

bar_colors = ['#1976D2', '#43A047', '#E53935', '#FB8C00', '#8E24AA']

df_sorted['Acc Teste'].plot(kind='barh', ax=axes[0],
                             color=bar_colors[:len(df_sorted)],
                             edgecolor='white')
axes[0].set_xlabel('Acurácia')
axes[0].set_title('Acurácia no Conjunto de Teste')
axes[0].set_xlim([0.9, 1.0])

df_sorted['Tempo (s)'].plot(kind='barh', ax=axes[1],
                             color=bar_colors[:len(df_sorted)],
                             edgecolor='white')
axes[1].set_xlabel('Tempo de Treino (s)')
axes[1].set_title('Tempo de Treinamento')

plt.tight_layout()
plt.show()

---
## 7. 💻 Exemplo Prático Completo: Regressão com Early Stopping

### Dataset: Diabetes + XGBoost com Early Stopping

In [ ]:
# ─── XGBoost Regressor com Early Stopping ────────────────────────────────────
diab = load_diabetes()
Xd, yd = diab.data, diab.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.2, random_state=42)
Xd_tr, Xd_val, yd_tr, yd_val = train_test_split(Xd_tr, yd_tr, test_size=0.15, random_state=42)

modelos_reg = {
    'Random Forest':   RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'GBM sklearn':     GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                  max_depth=4, subsample=0.8, random_state=42),
    'HistGBM sklearn': HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                                      random_state=42),
}

try:
    modelos_reg['XGBoost'] = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05,
        max_depth=4, subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=20, random_state=42, n_jobs=-1
    )
except: pass

print(f"{'Modelo':<22} {'RMSE':>8} {'R²':>8}")
print("-" * 42)
for nome, modelo in modelos_reg.items():
    if nome == 'XGBoost':
        try:
            modelo.fit(Xd_tr, yd_tr, eval_set=[(Xd_val, yd_val)], verbose=False)
            print(f"  XGBoost parou em {modelo.best_iteration} iterações (early stopping)")
        except Exception as e:
            modelo.fit(Xd_tr, yd_tr)
    else:
        modelo.fit(Xd_tr, yd_tr)

    preds = modelo.predict(Xd_te)
    rmse  = np.sqrt(mean_squared_error(yd_te, preds))
    r2    = r2_score(yd_te, preds)
    print(f"{nome:<22} {rmse:>8.2f} {r2:>8.4f}")

---
## 8. 🔍 Importância de Features e SHAP

Gradient Boosting também fornece importâncias de features. O XGBoost adiciona tipos adicionais:

| Tipo         | Descrição                                                |
|--------------|----------------------------------------------------------|
| `weight`     | Número de vezes que a feature foi usada em splits        |
| `gain`       | Ganho médio de cada split com a feature                  |
| `cover`      | Cobertura média (nº amostras afetadas) por split         |

**SHAP** (SHapley Additive exPlanations) é o método mais robusto e teoricamente fundamentado.

In [ ]:
# ─── Importância de Features: GBM vs RF ──────────────────────────────────────
bc = load_breast_cancer()
Xb_all, yb_all = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(
    Xb_all, yb_all, test_size=0.25, stratify=yb_all, random_state=42
)

rf_bc  = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xb_tr, yb_tr)
gbm_bc = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                     max_depth=3, random_state=42).fit(Xb_tr, yb_tr)

top_k = 10
names = bc.feature_names

imp_rf  = pd.Series(rf_bc.feature_importances_,  index=names).nlargest(top_k).sort_values()
imp_gbm = pd.Series(gbm_bc.feature_importances_, index=names).nlargest(top_k).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
imp_rf.plot(kind='barh',  ax=axes[0], color='#1976D2', edgecolor='white')
axes[0].set_title(f'Top {top_k} Features — Random Forest\nAcc={rf_bc.score(Xb_te, yb_te):.4f}')
imp_gbm.plot(kind='barh', ax=axes[1], color='#E53935', edgecolor='white')
axes[1].set_title(f'Top {top_k} Features — Gradient Boosting\nAcc={gbm_bc.score(Xb_te, yb_te):.4f}')
for ax in axes:
    ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()

---
## 9. 📊 Stochastic Gradient Boosting

**Stochastic GBM** (Friedman, 1999) melhora o GBM padrão adicionando aleatoriedade:

- `subsample < 1.0`: usa fração aleatória das amostras por árvore (como Bagging)
- `max_features < 1.0`: usa fração aleatória das features por split

Benefícios:
- Reduz overfitting
- Permite learning rates maiores
- Aproxima-se das vantagens do Random Forest

In [ ]:
# ─── Efeito do Subsample (Stochastic GBM) ────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                           stratify=y, random_state=42)

subsamples = [1.0, 0.8, 0.6, 0.4]
acc_tr_list, acc_te_list = [], []

for ss in subsamples:
    gbm_ss = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                         max_depth=3, subsample=ss, random_state=42)
    gbm_ss.fit(X_tr, y_tr)
    acc_tr_list.append(gbm_ss.score(X_tr, y_tr))
    acc_te_list.append(gbm_ss.score(X_te, y_te))

fig, ax = plt.subplots(figsize=(8, 5))
x_  = np.arange(len(subsamples))
w   = 0.35
ax.bar(x_ - w/2, acc_tr_list, width=w, label='Treino',  color='#FF5722', alpha=0.85)
ax.bar(x_ + w/2, acc_te_list, width=w, label='Teste',   color='#1976D2', alpha=0.85)
ax.set_xticks(x_)
ax.set_xticklabels([f'ss={s}' for s in subsamples])
ax.set_ylabel('Acurácia')
ax.set_title('Efeito do Subsample no GBM')
ax.set_ylim([0.85, 1.02])
ax.legend()

for i, (tr, te) in enumerate(zip(acc_tr_list, acc_te_list)):
    ax.text(i - w/2, tr + 0.002, f'{tr:.3f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, te + 0.002, f'{te:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
## 10. 🏁 Comparação Final: Quando Usar Cada Modelo?

| Critério                     | Random Forest             | Gradient Boosting             |
|------------------------------|---------------------------|-------------------------------|
| **Dados pequenos (<1k)**      | ✅ Robusto                | ⚠️ Pode overfitting           |
| **Dados médios (1k-100k)**    | ✅ Excelente              | ✅ Estado da arte             |
| **Dados grandes (>100k)**     | ✅ Bom, rápido            | ✅ XGBoost/LightGBM são rápidos|
| **Features com ruído**        | ✅ Resiliente             | ⚠️ Mais sensível              |
| **Dados com NaN**             | ⚠️ Precisa imputação     | ✅ XGBoost/HistGBM suportam nativamente|
| **Necessidade de tuning**     | ✅ Poucos parâmetros       | ⚠️ Mais parâmetros críticos   |
| **Interpretabilidade**        | ✅ MDI claro              | ✅ SHAP disponível            |
| **Performance máxima**        | 🥈 Muito bom              | 🥇 Geralmente melhor          |
| **Treino paralelo**           | ✅ Nativamente paralelo   | ⚠️ Sequencial (lento sem opt) |

### Regra Prática
- **Prototipagem rápida:** Random Forest
- **Competição / máxima performance:** XGBoost ou LightGBM
- **Pipeline scikit-learn nativo:** HistGradientBoosting

In [ ]:
# ─── Comparação Final em Dataset Sintético Desafiador ─────────────────────────
X_syn, y_syn = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, n_repeated=2, flip_y=0.05,
    random_state=42
)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_syn, y_syn,
                                                test_size=0.25, random_state=42)

modelos_final = {
    'Random Forest':   RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    'GBM sklearn':     GradientBoostingClassifier(n_estimators=200, learning_rate=0.1,
                                                   max_depth=4, subsample=0.8, random_state=42),
    'HistGBM':         HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1,
                                                       random_state=42),
}

try:
    modelos_final['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
except: pass

try:
    modelos_final['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        subsample=0.8, random_state=42, n_jobs=-1, verbose=-1
    )
except: pass

res_final = {}
for nome, modelo in modelos_final.items():
    modelo.fit(Xs_tr, ys_tr)
    cv5 = cross_val_score(modelo, X_syn, y_syn, cv=5, scoring='accuracy', n_jobs=-1)
    res_final[nome] = {'Teste': modelo.score(Xs_te, ys_te),
                        'CV Mean': cv5.mean(), 'CV Std': cv5.std()}

df_final = pd.DataFrame(res_final).T
print(df_final.sort_values('CV Mean', ascending=False).round(4))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 5))
df_plot = df_final.sort_values('CV Mean', ascending=True)
bars = ax.barh(df_plot.index, df_plot['CV Mean'],
               xerr=df_plot['CV Std'],
               color=['#1976D2','#43A047','#E53935','#FB8C00','#8E24AA'][:len(df_plot)],
               capsize=5, edgecolor='white', error_kw={'lw': 2})
ax.set_xlabel('Acurácia (Cross-Validation 5-fold)')
ax.set_title('Comparação Final: RF vs. Gradient Boosting')
ax.set_xlim([0.80, 1.0])
for bar, (nome, row) in zip(bars, df_plot.iterrows()):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{row['CV Mean']:.3f} ± {row['CV Std']:.3f}",
            va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## ✏️ Exercícios

### Exercício 1 — Entendendo os Resíduos

Usando o dataset `make_regression(n_samples=300, n_features=1, noise=20, random_state=0)`:

a) Treine um `GradientBoostingRegressor` com `n_estimators=50`, `learning_rate=0.3` e `max_depth=2`.  
b) Use `staged_predict` para obter as predições após 1, 5, 10, 25 e 50 iterações.  
c) Para cada iteração, plote em um gráfico: os dados reais, a curva de predição e os resíduos em barras.  
d) Observe como os resíduos diminuem. A partir de quantas iterações eles parecem estabilizar?

In [ ]:
# ── Exercício 1 ── Escreva seu código aqui ───────────────────────────────────

# Seu código:


---
### Exercício 2 — Regularização no GBM

Use `load_breast_cancer()` e explore o efeito da regularização:

a) Treine `GradientBoostingClassifier` com as seguintes combinações e registre acurácia de treino e teste:

| Experimento | `n_estimators` | `learning_rate` | `max_depth` | `subsample` |
|-------------|---------------|-----------------|-------------|-------------|
| A (default) | 100           | 0.1             | 3           | 1.0         |
| B (mais árvores) | 500      | 0.1             | 3           | 1.0         |
| C (overfitting) | 200        | 0.5             | 8           | 1.0         |
| D (regularizado) | 200       | 0.05            | 3           | 0.7         |

b) Qual configuração gerou maior diferença entre treino e teste (overfitting)?  
c) Compare os resultados com um `RandomForestClassifier(n_estimators=200)`.

In [ ]:
# ── Exercício 2 ── Escreva seu código aqui ───────────────────────────────────

# Seu código:


---
### Exercício 3 — Learning Rate × n_estimators

Usando `load_diabetes()`:

a) Para cada combinação abaixo, treine um `GradientBoostingRegressor` e registre o RMSE no teste:
   - (η=0.5, M=50), (η=0.1, M=250), (η=0.05, M=500), (η=0.01, M=2500)
   - Observe: η × M ≈ constante nas 4 configurações

b) Os RMSEs são similares? Há alguma diferença?  
c) Qual configuração foi mais rápida para treinar? Use `time.time()` para medir.  
d) **Conclusão:** como escolher entre um learning rate baixo com muitas árvores vs. learning rate alto com poucas?

In [ ]:
# ── Exercício 3 ── Escreva seu código aqui ───────────────────────────────────
import time

# Seu código:


---
### Exercício 4 — Desafio: Pipeline Completo com XGBoost / HistGBM

a) Gere um dataset com `make_classification(n_samples=3000, n_features=25, n_informative=12, n_redundant=5, flip_y=0.08, random_state=99)`.  
b) Divida em treino (70%), validação (15%) e teste (15%).  
c) Treine um `HistGradientBoostingClassifier` (ou XGBoost se disponível) com early stopping usando o conjunto de validação.  
d) Aplique `GridSearchCV` com `learning_rate` ∈ {0.05, 0.1, 0.2} e `max_depth` ∈ {3, 4, 6}.  
e) Reporte a matriz de confusão e o classification report para o melhor modelo no conjunto de teste.  
f) **Bônus:** Compare o resultado com um Random Forest otimizado e discuta qual modelo você escolheria em produção.

In [ ]:
# ── Exercício 4 ── Escreva seu código aqui ───────────────────────────────────

# Seu código:


---
## 💡 Gabarito Comentado

> **Atenção:** Tente resolver os exercícios antes de consultar! 👇

In [ ]:
# ── Gabarito Exercício 1 ─────────────────────────────────────────────────────
X_r1, y_r1 = make_regression(n_samples=300, n_features=1, noise=20, random_state=0)
X_r1_tr, X_r1_te, y_r1_tr, y_r1_te = train_test_split(X_r1, y_r1,
                                                         test_size=0.2, random_state=0)

gbm1 = GradientBoostingRegressor(n_estimators=50, learning_rate=0.3,
                                   max_depth=2, random_state=0)
gbm1.fit(X_r1_tr, y_r1_tr)

staged_iters = [1, 5, 10, 25, 50]
all_preds    = list(gbm1.staged_predict(X_r1_te))

fig, axes = plt.subplots(2, len(staged_iters), figsize=(20, 8))
X_line = np.linspace(X_r1.min(), X_r1.max(), 300).reshape(-1, 1)
staged_test_line = list(gbm1.staged_predict(X_line))

for col, it in enumerate(staged_iters):
    pred_line = staged_test_line[it - 1]
    pred_te   = all_preds[it - 1]
    residuos  = y_r1_te - pred_te
    rmse      = np.sqrt(np.mean(residuos**2))

    axes[0, col].scatter(X_r1_te, y_r1_te, s=20, alpha=0.5, color='#607D8B')
    axes[0, col].plot(X_line, pred_line, 'r-', lw=2)
    axes[0, col].set_title(f'm={it}\nRMSE={rmse:.1f}')

    axes[1, col].bar(range(len(residuos)), residuos,
                     color=['#F44336' if r < 0 else '#4CAF50' for r in residuos],
                     width=0.8, alpha=0.7)
    axes[1, col].axhline(0, color='black', lw=1)
    axes[1, col].set_ylim([-100, 100])

axes[0, 0].set_ylabel('y')
axes[1, 0].set_ylabel('Resíduo')
plt.suptitle('GBM Passo a Passo: Predição e Resíduos', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gabarito Exercício 2 ─────────────────────────────────────────────────────
bc2  = load_breast_cancer()
Xb2, yb2 = bc2.data, bc2.target
Xb2_tr, Xb2_te, yb2_tr, yb2_te = train_test_split(Xb2, yb2, test_size=0.25,
                                                     stratify=yb2, random_state=42)

configs = {
    'A - Default':       dict(n_estimators=100, learning_rate=0.1,  max_depth=3, subsample=1.0),
    'B - Mais árvores':  dict(n_estimators=500, learning_rate=0.1,  max_depth=3, subsample=1.0),
    'C - Overfitting':   dict(n_estimators=200, learning_rate=0.5,  max_depth=8, subsample=1.0),
    'D - Regularizado':  dict(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.7),
}

print(f"{'Config':<22} {'Treino':>8} {'Teste':>8} {'Gap':>8}")
print("-" * 50)
for nome, params in configs.items():
    m = GradientBoostingClassifier(**params, random_state=42)
    m.fit(Xb2_tr, yb2_tr)
    tr, te = m.score(Xb2_tr, yb2_tr), m.score(Xb2_te, yb2_te)
    print(f"{nome:<22} {tr:>8.4f} {te:>8.4f} {tr-te:>8.4f}")

# RF para comparação
rf2 = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xb2_tr, yb2_tr)
print(f"{'Random Forest':22} {rf2.score(Xb2_tr,yb2_tr):>8.4f} {rf2.score(Xb2_te,yb2_te):>8.4f}")

In [ ]:
# ── Gabarito Exercício 3 ─────────────────────────────────────────────────────
diab3 = load_diabetes()
Xd3, yd3 = diab3.data, diab3.target
Xd3_tr, Xd3_te, yd3_tr, yd3_te = train_test_split(Xd3, yd3, test_size=0.2, random_state=42)

combos = [(0.5, 50), (0.1, 250), (0.05, 500), (0.01, 2500)]

print(f"{'eta':>6} {'M':>6} {'RMSE':>8} {'Tempo(s)':>10}")
print("-" * 36)
for eta, M in combos:
    t0 = time.time()
    m = GradientBoostingRegressor(n_estimators=M, learning_rate=eta,
                                   max_depth=3, random_state=42)
    m.fit(Xd3_tr, yd3_tr)
    t  = time.time() - t0
    rm = np.sqrt(mean_squared_error(yd3_te, m.predict(Xd3_te)))
    print(f"{eta:>6.3f} {M:>6} {rm:>8.2f} {t:>10.3f}")

print("""
Análise:
- Os RMSEs são similares, confirmando que eta * M é o driver do desempenho.
- LR alto com poucas árvores: mais rápido, mas menos refinado.
- LR baixo com muitas árvores: melhor generalização (geralmente), mas custoso.
- Recomendação: use LR baixo com early stopping para melhor resultado prático.
""")

In [ ]:
# ── Gabarito Exercício 4 ─────────────────────────────────────────────────────
X4, y4 = make_classification(n_samples=3000, n_features=25, n_informative=12,
                              n_redundant=5, flip_y=0.08, random_state=99)

# Divisão 70/15/15
X4_tr, X4_tmp, y4_tr, y4_tmp = train_test_split(X4, y4, test_size=0.30,  random_state=42)
X4_val, X4_te, y4_val, y4_te = train_test_split(X4_tmp, y4_tmp, test_size=0.50, random_state=42)

# HistGBM com GridSearch
param_hist = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [3, 4, 6],
}

gs_hist = GridSearchCV(
    HistGradientBoostingClassifier(max_iter=300, random_state=42),
    param_hist, cv=5, scoring='accuracy', n_jobs=-1
)
gs_hist.fit(X4_tr, y4_tr)

print("Melhores parâmetros (HistGBM):", gs_hist.best_params_)
melhor = gs_hist.best_estimator_
y4_pred = melhor.predict(X4_te)

print("\nClassification Report:")
print(classification_report(y4_te, y4_pred))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y4_te, y4_pred)).plot(ax=ax, cmap='Blues')
ax.set_title(f'Melhor HistGBM — Acc={accuracy_score(y4_te, y4_pred):.3f}')
plt.tight_layout()
plt.show()

# Bônus: comparação com RF
rf4 = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rf4.fit(X4_tr, y4_tr)
print(f"\nRandom Forest Acc: {rf4.score(X4_te, y4_te):.4f}")
print(f"HistGBM Acc:       {accuracy_score(y4_te, y4_pred):.4f}")

---
## 📌 Resumo das Duas Aulas

### Aula 01 — Random Forest
| Conceito | Essência |
|----------|----------|
| Bootstrap + Agregação | Reduz variância via paralelismo |
| Feature aleatória por nó | Descorrelaciona as árvores |
| OOB Score | Validação gratuita sem CV |
| Feature Importance | MDI e Permutation |

### Aula 02 — Gradient Boosting
| Conceito | Essência |
|----------|----------|
| Gradiente funcional | Otimiza função de perda iterativamente |
| Pseudo-resíduo | Cada árvore corrige o erro anterior |
| Learning rate (η) | Controla o passo de cada atualização |
| Subsample | Adiciona aleatoriedade (Stochastic GBM) |
| XGBoost / LightGBM | Implementações de estado da arte |
| Early stopping | Evita overfitting automaticamente |

### A Grande Imagem
```
Dados → Random Forest (baseline rápido, robusto)
      ↓ precisa de mais performance?
      → Gradient Boosting (XGBoost / LightGBM) + tuning
      ↓ precisa interpretar?
      → SHAP Values + Feature Importance
```
